# MRT-Auswertung: Phasenfarben und Signalanteile

In diesem Notebook untersuchen Sie eine Wasser-/Ölflasche auf einer einzelnen Schichtebene. Das Laden, Prüfen und Anzeigen der Daten ist vorbereitet. **Sie ergänzen nur die Rechenschritte für den Phasen-Magnitude-Blend, die pixelweisen größeren und kleineren Modellanteile und die B₀-Frequenzkarte aus der Triple-Echo-Aufnahme.**

Lesen Sie vor dem Start die [README-Anleitung](README.md). Arbeiten Sie dann von oben nach unten mit **Shift + Enter**. Ersetzen Sie in den Aufgaben `...` durch Ihren Code und führen Sie den jeweiligen Block erst danach aus. Antworten tragen Sie per Doppelklick in die Textzellen ein. Speichern Sie regelmäßig.

Am Ende haben Sie ein rot-blaues Gesamtbild, eine Prozentkarte, eine B₀-Frequenzkarte und eine kurze Erklärung ihrer Aussagegrenzen.

## Vorbereitung

Führen Sie bei Bedarf die Installationszelle aus. Starten Sie danach den Kernel neu und führen Sie anschließend die **Importzelle darunter** aus. Sind die Bibliotheken bereits installiert, beginnen Sie direkt bei den Importen.

In [ ]:
%pip install -r requirements-jupyterhub.txt

In [ ]:
from pathlib import Path
import numpy as np
import pydicom
import matplotlib.pyplot as plt
import mrt_hilfen as mh

print("Bibliotheken geladen.")

## Daten laden und verstehen

| Begriff | Bedeutung in diesem Notebook |
| --- | --- |
| Echo / TE | Ein Bild wird zu einer bestimmten Echozeit TE aufgenommen. |
| Magnitude | Die Stärke des gemessenen Signals, dargestellt als Helligkeit. |
| Phase | Die Winkellage des Signals; hier nach dem Import in Radiant (rad). |
| IP / OP | Wasser und Fett schwingen ungefähr gleichphasig / gegenphasig. |
| DICOM-Metadaten | Begleitinformationen wie Echozeit, Bildtyp und Bildgeometrie. |

Die erste Datei legt die Schichtebene fest. Der Katalog sucht dazu passende Bilder. **Echonummer und IP/OP sind nicht dasselbe:** Entscheidend sind die tatsächliche TE und der Wasser-Fett-Frequenzabstand.

**Beispieldaten:** Voreingestellt ist die Wasser-/Ölflasche in `daten/beispiel/`. Alternativ können Sie das Knee-Phantom in `daten/knee/` verwenden: Setzen Sie dazu in der folgenden Zelle `datenordner = "daten/knee"` und die Referenzdatei auf `knee_dual_echo2_magnitude.dcm`. Beide Datensätze enthalten Dual- und Triple-Echo-Aufnahmen mit denselben Echozeiten; die Aufnahmenamen `T1W_FFE dual echo` und `T1W_FFE triple echo` sind identisch. Die [Dateiübersicht des Knee-Phantoms](daten/knee/README.md) beschreibt die Schichtauswahl.

In [ ]:
# Beispieldaten: Wasser-/Ölflasche (Voreinstellung)
datenordner = "daten/beispiel"
referenzdatei = Path(datenordner) / "wasser_oel_dual_echo2_magnitude.dcm"

# Alternativ: Knee-Phantom. Dazu die beiden Zeilen oben auskommentieren und diese aktivieren.
# datenordner = "daten/knee"
# referenzdatei = Path(datenordner) / "knee_dual_echo2_magnitude.dcm"

dicom = pydicom.dcmread(referenzdatei)
katalog = mh.katalog(datenordner, referenz=dicom)
auswahl = mh.echo_auswahl(katalog, "T1W_FFE dual echo")

Beim Beispiel ist das frühe Echo näherungsweise OP, das späte IP. Lassen Sie die Voreinstellung für den ersten Durchlauf stehen. Nach einer Änderung der Auswahl müssen Sie die folgende Zelle und alle nachfolgenden Zellen erneut ausführen.

Die Hilfsdatei prüft, ob die Bilder zur selben Aufnahme und Schichtebene gehören. Sie berechnet die Aufgaben dieses Notebooks nicht für Sie.

In [ ]:
op, ip, meta_op, meta_ip = mh.lade_paar(auswahl)
phase_op, meta_phase_op = mh.phase_zu_magnitude(katalog, meta_op)
phase_ip, meta_phase_ip = mh.phase_zu_magnitude(katalog, meta_ip)

print("Bildform [Zeilen, Spalten]:", ip.shape)
print("TE OP / IP [ms]:", meta_op["echozeit_ms"], meta_ip["echozeit_ms"])
print("Einheit der Phasen:", meta_phase_op["einheit"], meta_phase_ip["einheit"])

### Magnitude und Phase vergleichen

Führen Sie die folgenden vier Darstellungszellen einzeln aus. **Jedes Bild erhält eine eigene Abbildung.** Vergleichen Sie jeweils dieselben Bildorte.

Die Magnitudenbilder verwenden dieselbe Grauwertskala, die Phasenbilder dieselbe Winkelskala. Helligkeit ist keine Wasser-/Fett-Kennzeichnung. Die Phase im signalarmen Hintergrund ist unzuverlässig.

**IP-Magnitude** · Signalstärke beim näherungsweise gleichphasigen Echo.

In [ ]:
signalgrenze = mh.fenster(ip, op)[1]
fig_ip_magnitude = mh.bild(ip, "IP-Magnitude", meta_ip, (0, signalgrenze))

**OP-Magnitude** · Dieselbe Grauwertskala wie beim IP-Bild. Achten Sie auf Signalverluste.

In [ ]:
fig_op_magnitude = ...

**IP-Phase** · Die Farbe zeigt den Signalwinkel in Radiant.

In [ ]:
fig_ip_phase = mh.bild(phase_ip, "IP-Phase [rad]", meta_phase_ip, (-np.pi, np.pi), "twilight")

**OP-Phase** · Dieselbe Winkelskala wie beim IP-Phasenbild.

In [ ]:
fig_op_phase = ...

## Physikalische Grundlage

**IP bedeutet nicht „nur Wasser“.** Im idealen, hintergrundphasenkorrigierten Modell gilt $S_\mathrm{IP}=W+F$ und $S_\mathrm{OP}=W-F$. Ein Magnitudenbild zeigt nur den Betrag:

$$M_\mathrm{IP}=W+F,\qquad M_\mathrm{OP}=|W-F|.$$

Die Magnituden verlieren das Vorzeichen von $W-F$. Deshalb liefern sie zunächst einen größeren und einen kleineren Modellanteil, ohne deren Wasser-/Fett-Zuordnung festzulegen.

Für das Farbbild verwenden wir separat die gemessene Phasendifferenz zwischen OP und IP. **Rot und Blau kennzeichnen ausschließlich Phasenlagen.** Die Farben erhalten keine feste Wasser-/Öl-Bedeutung; dadurch lassen sich eigene Aufnahmen ohne zusätzliche Kalibrierdatei verwenden.

Die Beispiel-Echozeiten sind nur näherungsweise IP und OP. Die vorbereitete Kontrolle berechnet die chemischen Umdrehungen aus TE und der Skriptannahme von 3,5 ppm. Eine ganze Umdrehungszahl passt zu IP, eine halbe zu OP. Das ist eine Modellprüfung, keine Spektralmessung.

In [ ]:
F0 = meta_ip["referenz_mhz"]
chemie_hz = ... * F0  #Skript shift ppm × MHz = Hz oder auch der aus dem Experiment 1
OP_TE = meta_op["echozeit_ms"]
IP_TE = meta_ip["echozeit_ms"]


Umdrehung_IP = IP_TE * ... / 1000
Umdrehung_OP = ...
print("Chemische Umdrehungen OP / IP:", Umdrehung_OP, Umdrehung_IP)

### Welche Pixel sind auswertbar?

Eine automatische **Signalmaske** schließt schwaches Signal aus. Ein Pixel ist nur gültig, wenn beide Echos genügend Signal und endliche Werte besitzen. Die Schwelle beträgt jeweils 15 % des 99. Perzentils; sie ist eine einfache Heuristik.

Die Maske wird bereitgestellt. Außerhalb der Maske steht in den Rechenarrays `NaN`: „kein auswertbarer Zahlenwert“. Sie müssen keine Bildbereiche von Hand auswählen.

In [ ]:
signal_ip = np.isfinite(ip) & (ip > 0.15 * np.percentile(ip, 99))
maske = (signal_ip & np.isfinite(op) & np.isfinite(phase_ip) & np.isfinite(phase_op)
         & (op > 0.15 * np.percentile(op, 99)))
if not maske.any():
    raise ValueError("Kein gemeinsamer signalstarker Bereich in beiden Echos.")
ip_signal = np.where(maske, ip, np.nan)
op_signal = np.where(maske, op, np.nan)
print("Auswertbare Pixel:", int(maske.sum()))

## 12d · Phase als Farbe, Magnitude als Helligkeit

### Aufgabe 1: Die relative Phase berechnen

Ziehen Sie die IP-Phase von der OP-Phase ab. Winkel wiederholen sich nach $2\pi$. Führen Sie die Differenz deshalb auf den Bereich von $-\pi$ bis $\pi$ zurück:

$$\Delta\phi=\arg\{\exp[i(\phi_\mathrm{OP}-\phi_\mathrm{IP})]\}.$$

**Python-Hilfe:** `np.angle(z)` liefert den Winkel einer komplexen Zahl, `np.exp(...)` die Exponentialfunktion; `1j` ist die imaginäre Einheit. Ergänzen Sie die eine Rechenzeile. Es wird kein festgelegter Wasserbereich und kein vorgegebener Phasenoffset benötigt.

In [ ]:
delta_phase = ...

### Aufgabe 2: Farbe und Helligkeit zusammensetzen

Berechnen Sie zwei **Farbgewichte**: $r=(1-\cos\Delta\phi)/2$ und $b=1-r$. Bei gleicher Phase wird das Bild blau, bei entgegengesetzter Phase rot. Die Farben sind neutrale Darstellungsfarben und keine Stoffkennzeichnung.

Die Helligkeit $h$ ist die IP-Magnitude geteilt durch `signalgrenze`. Begrenzen Sie **nur diese Anzeigegröße** mit `np.clip(..., 0, 1)`. Multiplizieren Sie das rote bzw. blaue Gewicht mit $h$:

$$\mathrm{RGB}=(h\,r,\;0,\;h\,b).$$

Ergänzen Sie die fünf Ausdrücke. Das Zusammensetzen der Farbkanäle und die Darstellung sind vorbereitet.

In [ ]:
gewicht_rot = ...
gewicht_blau = ...
helligkeit = ...
rot = ...
blau = ...

rgb = np.stack((rot, np.zeros_like(rot), blau), axis=-1)
rgb[~maske] = 0
# IP-Signal, aber unzuverlässige Phase: grau statt einer Phasenfarbe.
grau = signal_ip & ~maske
rgb[grau] = helligkeit[grau, None]
fig_farbe = mh.phasenblend_anzeigen(rgb, meta_ip)

### So lesen Sie das Farbbild

| Farbe | Aussage |
| --- | --- |
| Blau | Die gemessene OP-/IP-Phasendifferenz liegt nahe null. |
| Rot | Die Phasendifferenz liegt nahe ±π. |
| Violett | Die Phase liegt dazwischen. |
| Grau | IP-Signal ist vorhanden, die Phase ist hier nicht auswertbar. |
| Schwarz | Zu wenig IP-Signal. |

Die beiden Farben zeigen Phasenunterschiede, **keine eindeutige Wasser-/Öl-Verteilung**. Ein Farbgewicht ist auch kein Signal- oder Mengenanteil. Feldabweichungen und echoabhängige Phasenfehler beeinflussen die Farben. Bei anderen Aufnahmen dürfen dieselben Flüssigkeiten deshalb anders eingefärbt sein.

Im nächsten Schritt berechnen Sie die Modellanteile direkt aus den Magnituden. Diese Rechnung verwendet die Farbdarstellung nicht.

## 13 · Größeren und kleineren Signalanteil pro Pixel berechnen

### Aufgabe 3: Die Magnituden auswerten

Aus dem vereinfachten Magnitudenmodell folgen ein größerer Anteil $A$ und ein kleinerer Anteil $B$:

$$A=\frac{M_\mathrm{IP}+M_\mathrm{OP}}{2},\qquad
B=\frac{M_\mathrm{IP}-M_\mathrm{OP}}{2},\qquad
p_B=100\,\frac{B}{A+B}.$$

Berechnen Sie die drei Größen mit `ip_signal` und `op_signal`. Die Rechnung erfolgt für jedes gültige Pixel. Sie benötigt keinen Phasenbezug.

**A ist nicht automatisch Wasser und B nicht automatisch Fett.** Im idealen Modell ist A der größere und B der kleinere der beiden Beiträge. Wasser und Fett können ihre Rollen tauschen, ohne die Magnituden zu ändern. Eine Stoffzuordnung würde zusätzliches Wissen erfordern und wird hier nicht vorgenommen.

Im idealen Modell liegt $p_B$ zwischen 0 und 50 %. Lassen Sie negative Signale oder Werte außerhalb dieses Intervalls stehen; sie zeigen Grenzen des Modells. Verwenden Sie keinen Betrag und kein Abschneiden.

In [ ]:
anteil_a = ...
anteil_b = ...
signalanteil_b_prozent = ...

fig_signalanteile = mh.signalanteile_anzeigen(anteil_a, anteil_b, signalanteil_b_prozent, maske, meta_ip)

### Ergebnisse prüfen

Die ersten beiden Karten zeigen die Modellanteile A und B. Die dritte zeigt den Signalanteil B in Prozent. **Orange** markiert negative Signale oder Prozentwerte außerhalb von 0 bis 50; **Hellgrau** nicht ausgewertete Pixel.

Die folgende Kontrolle prüft die Bildform, die Farbgewichte und die Beziehungen A + B = IP sowie A − B = OP. Sie zählt außerdem Pixel mit einer Modellverletzung. Daraus wird keine Stoffkonzentration der Flasche abgeleitet.

In [ ]:
assert delta_phase.shape == ip.shape
assert rgb.shape == (*ip.shape, 3)
assert np.isfinite(rgb).all() and np.all((rgb >= 0) & (rgb <= 1))
np.testing.assert_allclose((gewicht_rot + gewicht_blau)[maske], 1)
np.testing.assert_allclose((anteil_a + anteil_b)[maske], ip[maske])
np.testing.assert_allclose((anteil_a - anteil_b)[maske], op[maske])
assert np.isfinite(signalanteil_b_prozent[maske]).all()
assert np.isnan(signalanteil_b_prozent[~maske]).all()
modellverletzung = maske & ((anteil_a < 0) | (anteil_b < 0))
modellverletzung_prozent = 100 * modellverletzung.sum() / maske.sum()
print("Gültige Pixel:", int(maske.sum()))
print("Davon Pixel mit Modellverletzung [%]:", float(modellverletzung_prozent))

### Kurzantworten

1. Warum kann ein reiner Ölbereich im IP-Magnitudenbild ebenfalls hell sein?
2. Was unterscheidet das rote Farbgewicht `gewicht_rot` vom Signalanteil `signalanteil_b_prozent`?
3. Warum darf man A und B nicht ohne Weiteres Wasser und Fett zuordnen oder die Farben als Stoffkennzeichnung lesen?

*Ihre Antworten: …*

## 14 · B₀-Karte aus der Triple-Echo-Aufnahme

Wir vergleichen **Echo 3 mit Echo 1**. Wenn sich die chemische Phasenentwicklung um eine ganze Anzahl Umdrehungen weiterbewegt hat, ist die relative Phase wieder ungefähr null. Die absolute Phase des dritten Bildes muss nicht null sein. Lokale Feldabweichungen erzeugen eine zusätzliche Phasendifferenz; daraus wird die Frequenzabweichung berechnet.

Das Laden ist vorbereitet. Verwenden Sie die äußeren Phasenbilder derselben Triple-Echo-Aufnahme. Die tatsächlichen Echozeiten werden aus DICOM gelesen.

In [ ]:
b0_auswahl = mh.echo_auswahl(katalog, "... AUFNAHME ...", modus="b0")

In [ ]:
b0_phase_1, b0_phase_3, b0_meta_1, b0_meta_3 = mh.lade_paar(b0_auswahl)


B0_TE_ECHO_1 = ...
B0_TE_ECHO_3 = ...

print("TE Echo 1 / Echo 3 [ms]:", B0_TE_ECHO_1, B0_TE_ECHO_3)
print("Chemische Umdrehungen zwischen den Echos:", chemie_hz * (B0_TE_ECHO_3 - B0_TE_ECHO_1) / 1000)

### Aufgabe 4: Phasendifferenz → Frequenzabweichung → Karte

Berechnen Sie den Echoabstand in **Sekunden**, die Phasendifferenz in **Radiant** und daraus die Frequenzabweichung in **Hz**:

$$\Delta TE=(TE_3-TE_1)/1000,\qquad
\Delta\phi=\arg\{\exp[i(\phi_3-\phi_1)]\},\qquad
\Delta f=\frac{\Delta\phi}{2\pi\,\Delta TE}.$$

Ergänzen Sie die drei Rechenzeilen. Die vorbereitete Anzeige zeichnet die gesamte Schicht **ohne Maske**. Im signalarmen Hintergrund ist die Phase verrauscht; dort hat die Karte keine verlässliche Feldaussage.

Die Phasendifferenz liegt zwischen $-\pi$ und $\pi$. Ohne räumliches Entfalten ist die Frequenz daher nur modulo $1/\Delta TE$ bestimmt; Sprünge können Phasenumbrüche sein. Außerdem ist die chemische Gleichphasigkeit im Beispiel nur näherungsweise erfüllt. Die Karte ist eine einfache B₀-Darstellung in Hz, keine absolut kalibrierte Feldmessung. Das Vorzeichen folgt der Annahme $\phi=\phi_0+2\pi\,\Delta f\,TE$.

In [ ]:
delta_te_b0_s = ...
delta_phase_b0 = ...
b0_hz = ...

b0_grenze_hz = 1 / (2 * delta_te_b0_s)
fig_b0 = mh.bild(b0_hz, "B₀: Frequenzabweichung [Hz] · ohne Maske / ohne Entfalten",
                 b0_meta_1, (-b0_grenze_hz, b0_grenze_hz), "RdBu_r")

## Eigene Messdaten verwenden

Laden Sie Ihre DICOM-Dateien in `daten/eigene/` hoch. Sie benötigen Magnitude und Phase beider Echos auf derselben Schichtebene. Ersetzen Sie in der Ladezelle den Datenordner, den Dateipfad und die Aufnahme durch Ihre eigenen Daten. Dateinamen und Aufnahmebezeichnungen dürfen von den Beispielen abweichen; die Paarprüfung nutzt DICOM-Metadaten.

Führen Sie danach alle folgenden Zellen erneut aus. Die Farbformel verarbeitet auch gedrehte Flaschenbilder und benötigt keine zusätzliche Kalibrierdatei. Rot und Blau kennzeichnen nur die gemessene Phasenlage. Prüfen Sie für die Magnitudenrechnung weiterhin anhand der tatsächlichen Echozeiten und Ihres Messprotokolls, ob die Echos näherungsweise IP und OP entsprechen.

Für die B₀-Aufgabe benötigen Sie zusätzlich die Phasenbilder der äußeren Triple-Echos. Wählen Sie dort Ihre Triple-Echo-Aufnahme und kontrollieren Sie den tatsächlichen Echoabstand sowie die chemische Gleichphasigkeit.

### Hintergrund

Die Übung zeigt einen qualitativen Phasen-Magnitude-Blend und eine vereinfachte Signalaufteilung. Für eine belastbare komplexe Wasser-/Fett-Trennung müssen unter anderem räumliche Phasenfehler berücksichtigt werden: [Berglund et al., 2011: Two-point Dixon method with flexible echo times](https://doi.org/10.1002/mrm.22679).

Die Hilfsdatei übernimmt DICOM-Import, Skalierung, Paarprüfung und Darstellung. **Die Rechenschritte für Phase, Farbgewichte, Helligkeit und Signalanteile stehen sichtbar in Ihrem Notebook.**

Die B₀-Frequenzkarte folgt dem Zusammenhang zwischen Phasendifferenz und Echoabstand: [SDCFlows-Dokumentation](https://www.nipreps.org/sdcflows/1.4/_modules/sdcflows/workflows/fit/fieldmap.html). Hier wird bewusst die einfache, nicht räumlich entfaltete Darstellung gezeigt.